# Zhong, Katkov & Tsodyks (2026), "Synaptic Theory of Chunking in Working Memory", on DANDI:000207

- **Paper.** doi:10.7554/elife.109538. The claims file (`claims/zhong-2026.yaml`) was written from the
  text under `paper/`, the eLife Reviewed Preprint version 1 (doi:10.7554/eLife.109538.1), with its
  Methods A to E and the legends of Figures 1 to 4 and S1 to S3.
- **Dataset.** DANDI:000207, "Data for: Neurons detect cognitive boundaries to structure episodic
  memories in humans", version **0.220216.0323** (doi:10.48324/dandi.000207/0.220216.0323). This is the
  version the paper cites (Methods E, "Data analyzed in Fig. 3 is downloaded from the DANDI Archive at
  https://dandiarchive.org/dandiset/000207/0.220216.0323"), so it is pinned. Two later versions exist:
  0.220721.1915 has the same files with the same sizes, and 0.230530.1822 renames them and changes
  some (one is compared below, in the data check). The files are streamed; nothing is downloaded.
- **What the dataset can test.** The dataset is the single-neuron recordings of Zheng et al. 2022
  (ref. [41] of the paper), which the paper reanalyses for Figure 3 and Figure S3. The simulations
  (Figures 1, 2, S1, S2), the derivation of Eq. (1) and the reanalysis of published recall tables
  (Figure 4) do not use these data.
- **Outside material.** The Methods of [41] (PMC8966433), for how boundary, event and onset cells
  are selected, and its Supplementary Table 2 (per-subject counts of those cells), transcribed into the
  helper as reference values. The dataset's example code (github.com/rutishauserlab/cogboundary-zheng at
  commit 60f7ece3c51fb424aaca57b9daa41d19a9ef768c) for how the boundary columns map to cuts.

| Claim | Tested here |
|---|---|
| link-zheng-table-s2 (added link claim) | tested |
| methods-subjects-included | tested |
| fig3a-dip-after-boundary | analysis run under every reading; blocked (selection of [41] not recovered) |
| fig3b-individual-subjects | analysis run; blocked with fig3a |
| fig3c-onset-control | analysis run; blocked with fig3a |
| fig1d-basic-capacity-four, fig1d-chunking-retrieves-six, eq1-new-magic-number, fig2b-hierarchical-retrieval, figs2-delayed-hebbian-binding | blocked: simulations and a derivation, no data |
| fig4a-universal-slope, fig4b-plateau-near-eight, fig4b-three-languages | blocked: published recall tables of 1950 to 1988, not in any data archive |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chunking as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
t_start = time.time()
assets = C.list_assets()
sessions = [C.load_session(a) for a in assets]
print(f"dandiset {C.DANDISET} version {C.VERSION}: {len(assets)} assets, loaded in {time.time() - t_start:.0f} s")

## Data check

Every file is listed with what it holds. The units table has only `spike_times` and `electrodes`: no
column classifies a unit as a boundary, event or onset cell. The encoding table has, per clip,
`start_time`, `stop_time`, `fixcross_time`, `stimCategory` (0 NB, 1 SB, 2 HB, from the clip names
NB_, SB_, HB_), `boundary1_time` to `boundary3_time` and `Clip_name`.

In [ ]:
rows = []
for s in sessions:
    t, k, _ = s.cuts("all")
    st = np.concatenate(s.spikes)
    rel = t - s.clip_start[_]
    locs = pd.Series([l.split("_")[0] for l in s.unit_location]).value_counts().to_dict()
    rows.append(dict(
        file=s.path.split("/")[0], identifier=s.identifier, patient=s.patient, subject_id=s.subject_id,
        age=s.age, sex=s.sex, units=len(s.spikes), **{f"units_{k_}": v for k_, v in locs.items()},
        clips=len(s.category), NB=int((s.category == 0).sum()), SB=int((s.category == 1).sum()),
        HB=int((s.category == 2).sum()), cuts_SB=int((k == 1).sum()), cuts_HB=int((k == 2).sum()),
        HB_at_s=f"{rel[k == 2].min():.3f}-{rel[k == 2].max():.3f}",
        NB_at_s=f"{rel[k == 0].min():.2f}-{rel[k == 0].max():.2f}",
        clip_len_s=f"{(s.clip_stop - s.clip_start).min():.2f}-{(s.clip_stop - s.clip_start).max():.2f}",
        clips_from_s=round(float(s.clip_start.min()), 1), clips_to_s=round(float(s.clip_stop.max()), 1),
        spikes_from_s=float(st.min()), spikes_to_s=float(st.max()),
        on_trial_clock=C.spikes_on_trial_clock(s), recog=s.n_recog, timediscrim=s.n_timediscrim))
survey = pd.DataFrame(rows).fillna(0)
print(survey.to_string())
print()
print("subjects:", survey.patient.nunique(), " sessions:", len(survey), " units:", survey.units.sum(),
      " units by area:", {c: int(survey[c].sum()) for c in survey if c.startswith("units_")})
print("all units are in the medial temporal lobe:",
      all(l.split("_")[0] in ("amygdala", "hippocampus", "parahippocampus") for s in sessions for l in s.unit_location))
print("sessions whose spike times are not on the trial clock:", list(survey.loc[~survey.on_trial_clock, "identifier"]))
nb_half = np.concatenate([((s.clip_stop - s.clip_start) / 2)[s.category == 0] for s in sessions])
nb_rel = np.concatenate([(s.b[:, 0] - s.clip_start)[s.category == 0] for s in sessions])
print(f"NB virtual boundary relative to clip start: {nb_rel.min():.3f} to {nb_rel.max():.3f} s; half the NB clip length: "
      f"{nb_half.min():.2f} to {nb_half.max():.2f} s")
sorted_ok = all(np.all(np.diff(x) >= 0) for s in sessions for x in s.spikes)
print("spike times sorted within every unit:", sorted_ok)
rates = np.array([len(x) / (x[-1] - x[0]) for s in sessions for x in s.spikes if len(x) > 1])
print(f"mean firing rates: median {np.median(rates):.2f} Hz, range {rates.min():.3f} to {rates.max():.2f} Hz")

Recordings deposited more than once: every pair of units across and within files is compared on
their spike count and first and last spike times, and every pair of sessions on their clip order.

In [ ]:
keys = {}
dups = []
for s in sessions:
    for u, x in enumerate(s.spikes):
        key = (len(x), round(float(x[0]), 6), round(float(x[-1]), 6))
        if key in keys:
            dups.append((keys[key], (s.patient, u)))
        keys[key] = (s.patient, u)
print("units with identical spike count, first and last spike:", dups if dups else "none")
orders = {}
for s in sessions:
    orders.setdefault(tuple(np.round(s.clip_start - s.clip_start[0], 3)), []).append(s.patient)
print("sessions with identical clip timing:", [v for v in orders.values() if len(v) > 1] or "none")

**Clock.** The spikes of TWH138 (sub-19) lie between about -7.0e12 and -7.0e12 + 2967 s, while its
trial table runs from 1 to 2892 s; the two do not share a clock, and no offset in the file relates
them. For the other 18 sessions, the protocol's check: decode, from the population's spike counts,
the 1 s after clip onset from the 1 s before it (cross-validated logistic regression), with the true
onset times and with every onset shifted by 30 s, which breaks the tie between windows and onsets;
and the paired t statistic over clips of the population's summed count after minus before onset, also
at recognition and time-discrimination image onsets, and its largest value over shifts of -12 to 12 s.

In [ ]:
clock = []
for s in sessions:
    clock.append(dict(patient=s.patient, units=len(s.spikes), true=C.clock_decoding(s, 0.0),
                      shifted_30s=C.clock_decoding(s, 30.0), pop_t_true=C.clock_population_t(s, 0.0),
                      pop_t_shifted_30s=C.clock_population_t(s, 30.0),
                      recog_t=C.clock_population_t(s, 0.0, s.recog_start),
                      recog_t_shifted=C.clock_population_t(s, 30.0, s.recog_start),
                      timediscrim_t=C.clock_population_t(s, 0.0, s.timediscrim_start),
                      enc_max_abs_t_within_12s=max(abs(C.clock_population_t(s, sh)) for sh in np.arange(-12, 12.01, 0.25))))
clock = pd.DataFrame(clock)
print(clock.round(3).to_string())
ok = clock.dropna()
print(f"median accuracy, true onsets {ok.true.median():.3f}, shifted {ok.shifted_30s.median():.3f}; "
      f"sessions with true above shifted: {(ok.true > ok.shifted_30s).sum()} of {len(ok)}")
est = ok[ok.pop_t_true.abs() >= 3]
clock_ok = set(est.patient)
print(f"sessions where the encoding clock is shown (|population t| at least 3 at true clip onsets): {len(est)} of {len(ok)}: {sorted(clock_ok)}")
print("  not shown:", list(ok.loc[~ok.index.isin(est.index), "patient"]))
print(f"  with onsets shifted by 30 s: |t| at least 3 in {(ok.pop_t_shifted_30s.abs() >= 3).sum()} sessions (largest {ok.pop_t_shifted_30s.abs().max():.2f}); "
      f"recognition |t| at least 3 in {(ok.recog_t.abs() >= 3).sum()} sessions true, {(ok.recog_t_shifted.abs() >= 3).sum()} shifted")
both = ok[(ok.pop_t_true.abs() >= 3) & (ok.recog_t.abs() >= 3)]
only_rec = ok[(ok.pop_t_true.abs() < 3) & (ok.recog_t.abs() >= 3)]
print(f"  sessions responding at recognition onsets but not at encoding onsets: {list(only_rec.patient)}; at both: {list(both.patient)}")

**Versions.** The latest version (0.230530.1822) renames the files. For TWH120 (sub-14) its size
differs; the spike counts per unit are compared with the pinned version here.

In [ ]:
latest = [a for a in C.list_assets("0.230530.1822") if a["path"].startswith("sub-14/")][0]
s_new = C.load_session(latest, "0.230530.1822")
s_old = [s for s in sessions if s.path.startswith("sub-14/")][0]
n_old = np.array([len(x) for x in s_old.spikes]); n_new = np.array([len(x) for x in s_new.spikes])
print(f"TWH120: units {len(n_old)} (pinned) and {len(n_new)} (0.230530.1822); spikes {n_old.sum()} and {n_new.sum()}; "
      f"units whose count differs: {(n_old != n_new).sum()}; largest relative difference {np.max(np.abs(n_new - n_old) / n_old):.3f}")
print(f"encoding tables: clip starts differ by at most {np.max(np.abs(s_old.clip_start - s_new.clip_start)) * 1000:.3f} ms, "
      f"boundary times by at most {np.nanmax(np.abs(s_old.b - s_new.b)) * 1000:.3f} ms (NaN in {int(np.isnan(s_new.b).sum())} latest and {int(np.isnan(s_old.b).sum())} pinned values), categories identical {np.array_equal(s_old.category, s_new.category)}")
d_first = [x[0] - y[0] for x, y in zip(s_new.spikes, s_old.spikes) if len(x) and len(y)]
print(f"first spike times, latest minus pinned: {np.min(d_first) * 1000:.2f} to {np.max(d_first) * 1000:.2f} ms")
del s_new

## link-zheng-table-s2: the dataset's subjects are those of Supplementary Table 2 of [41]

Subjects are matched by the patient ID in each file's identifier (`P64CS_CBID12` is patient P64CS,
CBID 12) against the table's Patient ID and Subject ID; the match must agree on age and sex, and the
dataset's unit count, which holds only MTL units, can be at most the table's count of all cells.

In [ ]:
tab = C.zheng_table()
link = survey.set_index("patient")[["subject_id", "age", "sex", "units"]].rename(
    columns={"age": "nwb_age", "sex": "nwb_sex"}).join(tab, how="outer")
link["id_ok"] = link.subject_id.astype(float) == link.cbid.astype(float)
link["age_ok"] = link.nwb_age.str.extract(r"P(\d+)Y")[0].astype(float) == link.age.astype(float)
link["sex_ok"] = link.nwb_sex.str.upper() == link.sex
link["units_le_all"] = link.units <= link.all_cells
cols = ["subject_id", "cbid", "units", "all_cells", "boundary_mtl", "event_mtl", "onset", "id_ok", "age_ok", "sex_ok", "units_le_all"]
print(link[cols].to_string())
print(f"matched {link.id_ok.sum()} of {len(tab)} table subjects; age agrees {link.age_ok.sum()}, sex agrees {link.sex_ok.sum()}, "
      f"units at most all cells {link.units_le_all.sum()}")
print(f"table sums: all cells {tab.all_cells.sum()} (header says 985), boundary MTL {tab.boundary_mtl.sum()} (42), "
      f"event MTL {tab.event_mtl.sum()} (36), onset {tab.onset.sum()} (76); dataset units {survey.units.sum()} ([41]: 580 MTL)")

**Result: reproduced. All 19 subjects of Supplementary Table 2 of [41] with neurons are in the dataset under the same patient ID and CBID, with the same age and sex, and every file holds no more units than the table's count of all cells. The table's columns sum to 42, 36 and 76 as its header says, but its count of all cells sums to 943, not 985. The dataset holds 565 units, all in the MTL, where [41] reports 580 MTL neurons.**

## Selecting boundary, event and onset neurons (reading 1b of the claims file)

The dataset does not say which units are boundary, event or onset cells, and [41] names none (its
Supplementary Table 2 gives only counts per subject), so reading (1a) cannot be run on units. Reading
(1b) re-selects them by the Methods of [41], as the helper's docstring cites. Two details are open in
those Methods and both are run: whether the post-against-baseline test pairs the two windows of a trial
(`paired`), and whether "significantly greater in SB and HB than NB" means each of SB and HB against NB
(`literal`) or the two pooled (`pooled`). Boundary neurons of the paper are boundary (soft) and event
(hard) cells together (Methods E). TWH138 is left out, its spikes not being on the trial clock.

In [ ]:
usable = [s for s in sessions if C.spikes_on_trial_clock(s)]
t0 = time.time()
sel = {}
for paired in (True, False):
    sel[paired] = C.select_units(usable, paired=paired, which="all", seed=100 + int(paired))
print(f"classified {len(sel[True])} units of {len(usable)} sessions twice in {time.time() - t0:.0f} s")
per = []
for paired, d in sel.items():
    for pooled in (False, True):
        soft = d.soft_pooled if pooled else d.soft
        g = d.assign(bnd=soft | d.hard).groupby("patient", sort=False)
        per.append(pd.DataFrame(dict(soft=g.apply(lambda x: (x.soft_pooled if pooled else x.soft).sum()),
                                     hard=g.hard.sum(), onset=g.onset.sum())).add_prefix(
            f"{'paired' if paired else 'unpaired'}_{'pooled' if pooled else 'literal'}_"))
per = pd.concat(per, axis=1)
cmp = per.join(tab[["boundary_mtl", "event_mtl", "onset"]].add_prefix("table_"))
print(cmp.to_string())
print()
print("totals:", cmp.sum().to_dict())
for c in [c for c in cmp if c.endswith("_onset") and not c.startswith("table")]:
    r = np.corrcoef(cmp[c], cmp.table_onset)[0, 1]
    print(f"{c}: per-subject correlation with the table's onset counts r = {r:.2f}")
for c in [c for c in cmp if c.endswith("_soft")]:
    hard = c.replace("_soft", "_hard")
    r = np.corrcoef(cmp[c] + cmp[hard], cmp.table_boundary_mtl + cmp.table_event_mtl)[0, 1]
    print(f"{c[:-5]}: boundary plus event cells {int((cmp[c] + cmp[hard]).sum())} (table {int((cmp.table_boundary_mtl + cmp.table_event_mtl).sum())}), per-subject r = {r:.2f}")
n_units = len(sel[True])
print(f"p-value counts, paired reading, out of {n_units} units:",
      {c: int((sel[True][c] < 0.05).sum()) for c in ["pp_NB", "pp_SB", "pp_HB", "SB>NB", "HB>NB", "HB>SB", "SBHB>NB", "p_onset"]})

**Chance level of this selection.** [41] reports the fraction of MTL cells selected by chance, with the
boundary labels shuffled 1000 times: 2.11% for boundary cells and 2.26% for event cells (its Results and
Supplementary Table 4). The same is computed here with 20 shuffles, which is enough to tell a rate near
2% from one near 0.2% (1000 would take about 3 h).

In [ ]:
t0 = time.time()
soft_null, hard_null, n = C.shuffled_selection_rate(usable, n_shuffle=20, seed=7, paired=True)
print(f"{len(soft_null)} shuffles over {n} units in {time.time() - t0:.0f} s")
print(f"chance rate of boundary (soft) cells: mean {100 * soft_null.mean():.2f}% (range {100 * soft_null.min():.2f} to {100 * soft_null.max():.2f}%), [41]: 2.11%")
print(f"chance rate of event (hard) cells: mean {100 * hard_null.mean():.2f}% (range {100 * hard_null.min():.2f} to {100 * hard_null.max():.2f}%), [41]: 2.26%")
print(f"observed with true labels (paired, literal): soft {100 * sel[True].soft.mean():.2f}%, hard {100 * sel[True].hard.mean():.2f}%; [41]: 7.24% and 6.20%")

## methods-subjects-included

> Four of the eighteen subjects of Zheng et al. 2022 are left out of the Figure 3 analysis, because no
> onset-responsive or no cut-responsive neuron was detected in them, leaving 14.

The rule of Methods E is applied under each reading: (a) the classification of [41], through the
per-subject counts of its Supplementary Table 2 (boundary or event cells in the MTL, and clip-onset
cells), and (b) the re-selection above, under its four variants.

In [ ]:
n_enc = survey.patient.nunique()
n_clock = len(usable)
print(f"subjects with encoding sessions in the dataset: {n_enc}; with spike times on the trial clock: {n_clock}")
readings = {}
t = tab.copy()
t["cut_neurons"] = t.boundary_mtl + t.event_mtl
keep_a = t[(t.cut_neurons > 0) & (t.onset > 0)].index
readings["(a) [41] table, all 19 subjects"] = (len(t), list(keep_a), sorted(set(t.index) - set(keep_a)))
t18 = t.loc[[s.patient for s in usable]]
keep_a18 = t18[(t18.cut_neurons > 0) & (t18.onset > 0)].index
readings["(a) [41] table, 18 subjects on the trial clock"] = (len(t18), list(keep_a18), sorted(set(t18.index) - set(keep_a18)))
for c in [c for c in per if c.endswith("_soft")]:
    base = c[:-5]
    keep = per[(per[c] + per[base + "_hard"] > 0) & (per[base + "_onset"] > 0)].index
    readings[f"(b) {base}"] = (len(per), list(keep), sorted(set(per.index) - set(keep)))
subj = pd.DataFrame([dict(reading=k, subjects=v[0], kept=len(v[1]), excluded=", ".join(v[2]),
                          P64CS_kept="P64CS" in v[1], TWH120_kept="TWH120" in v[1]) for k, v in readings.items()])
print(subj.to_string())
included_a = list(keep_a18)

**Result: partially reproduced, on reading (a). Through the counts of Supplementary Table 2 of [41], the rule of Methods E leaves out exactly four subjects (TWH103, TWH109, TWH111, TWH125, which have no clip-onset cell) and keeps 14 of the 18 subjects whose spike times are on the trial clock (15 of all 19), P64CS and TWH120 among them. The dataset holds 19 subjects with encoding sessions, not 18; the count of 18 is matched only once TWH138, whose spike times are not on its trial clock, is set aside, which the paper does not say it did. Under reading (b), the re-selection by the Methods of [41], 5 to 9 of the 18 subjects are kept and P64CS is never among them.**

## fig3a-dip-after-boundary

> Aligned to movie cuts and averaged over subjects and trials, the firing of non-boundary neurons dips
> about 130 ms after the peak of boundary-neuron firing, then ramps up continuously, as the chunking
> model predicts.

Readings, from the criterion: bins of 10 or 25 ms, each without smoothing or with a 50 ms Gaussian;
z-score over the whole window or over -1 to 0 s; the window is -1 to 2 s; trials are the cuts of
clips with a cut. Two more readings for a detail the claims file leaves open: the trials are every SB
and HB cut (`all`, 105 per session, as [41] aligns to "boundaries") or only the first cut of each SB and
HB clip (`first`, 60 per session, for "each consisting of two episodes separated by a cut"). The
selection readings are those above. Subjects are those the rule of Methods E keeps under the same
selection reading.

In [ ]:
BINS = (0.010, 0.025)
SMOOTH = (None, 0.050)
ZREF = ("whole", "pre")
CUTSETS = ("all", "first")

def cut_times(s, cutset):
    t, k, _ = s.cuts(cutset)
    return t[k > 0]

# per-unit trial-averaged rates, computed once per session, cut set and bin
t0 = time.time()
unit_traces = {}
for s in usable:
    for cs in CUTSETS:
        for b in BINS:
            unit_traces[(s.patient, cs, b)] = C.unit_mean_rates(s, cut_times(s, cs), b)
    for b in BINS:
        unit_traces[(s.patient, "onset", b)] = C.unit_mean_rates(s, s.clip_start, b)
print(f"unit traces in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

def groups_for(selname):
    paired, pooled = selname.split("_")
    d = sel[paired == "paired"]
    soft = d.soft_pooled if pooled == "pooled" else d.soft
    return d.assign(bnd=soft | d.hard)

def subject_traces(patient, align, b, sm, zref, member):
    # member: boolean array over the subject's units (the group); returns z-scored group and rest traces
    R, centers = unit_traces[(patient, align, b)]
    g = C.smooth(R[member].mean(0), b, sm) if member.any() else None
    r = C.smooth(R[~member].mean(0), b, sm)
    return (C.zscore(g, centers, zref) if g is not None else None), C.zscore(r, centers, zref), centers

def group_average(patients, align, b, sm, zref, members):
    G, Rr = [], []
    for p in patients:
        g, r, centers = subject_traces(p, align, b, sm, zref, members[p])
        G.append(g); Rr.append(r)
    G, Rr = np.array(G), np.array(Rr)
    return G.mean(0), G.std(0), Rr.mean(0), Rr.std(0), centers, G, Rr

SELS = ["paired_literal", "paired_pooled", "unpaired_literal", "unpaired_pooled"]
rows = []
for selname in SELS:
    d = groups_for(selname)
    members = {p: x.bnd.values for p, x in d.groupby("patient", sort=False)}
    onset_m = {p: x.onset.values for p, x in d.groupby("patient", sort=False)}
    pats_all = [p for p in members if members[p].any() and onset_m[p].any()]
    for subset, pats in (("included", pats_all), ("clock shown", [p for p in pats_all if p in clock_ok])):
      if not pats:
        continue
      for cs in CUTSETS:
        for b in BINS:
            for sm in SMOOTH:
                for zref in ZREF:
                    gm, gs, rm, rs, centers, _, _ = group_average(pats, cs, b, sm, zref, members)
                    m = C.dip_ramp(gm, rm, centers)
                    tol = max(13.0, 1000 * b)
                    rows.append(dict(selection=selname, subset=subset, cuts=cs, bin_ms=int(1000 * b), smooth_ms=int(1000 * sm) if sm else 0,
                                     zref=zref, subjects=len(pats), bnd_units=int(sum(members[p].sum() for p in pats)),
                                     peak_ms=round(1000 * m["peak_t"]), dip_ms=round(1000 * m["dip_t"]), lag_ms=round(m["lag_ms"]),
                                     depth=round(m["depth"], 3), dip=m["dip"], slope=round(m["slope"], 3), ramp=m["ramp"],
                                     lag_ok=abs(m["lag_ms"] - 130) <= tol,
                                     verdict=("reproduced" if m["dip"] and m["ramp"] and abs(m["lag_ms"] - 130) <= tol
                                              else "partial" if m["dip"] and m["ramp"] else "not")))
fig3a = pd.DataFrame(rows)
print(fig3a.to_string())
print()
for subset, part in fig3a.groupby("subset"):
    print(f"{subset}: verdicts over {len(part)} readings {part.verdict.value_counts().to_dict()}; lag {part.lag_ms.min()} to {part.lag_ms.max()} ms; "
          f"readings with dip and ramp {int((part.dip & part.ramp).sum())}; subjects {part.subjects.min()} to {part.subjects.max()}")

In [ ]:
# the traces under one reading (paired, literal, all cuts, 10 ms, 50 ms smoothing, whole-window z)
d = groups_for("paired_literal")
members = {p: x.bnd.values for p, x in d.groupby("patient", sort=False)}
onset_m = {p: x.onset.values for p, x in d.groupby("patient", sort=False)}
pats = [p for p in members if members[p].any() and onset_m[p].any()]
gm, gs, rm, rs, centers, G, Rr = group_average(pats, "all", 0.010, 0.050, "whole", members)
fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
for a, m_, s_, name in ((ax[0], gm, gs, "boundary neurons"), (ax[1], rm, rs, "non-boundary neurons")):
    a.plot(centers, m_); a.fill_between(centers, m_ - s_, m_ + s_, alpha=0.3); a.axvline(0, ls="--", c="k")
    a.set_ylabel("z-scored rate"); a.set_title(f"{name}, {len(pats)} subjects")
ax[1].set_xlabel("time from cut (s)")
plt.tight_layout(); plt.show()
print(f"plotted ranges: boundary mean {gm.min():.2f} to {gm.max():.2f} (shade {np.min(gm - gs):.2f} to {np.max(gm + gs):.2f}); "
      f"non-boundary mean {rm.min():.2f} to {rm.max():.2f} (shade {np.min(rm - rs):.2f} to {np.max(rm + rs):.2f}); nothing outside the axes")
print("subjects:", pats, " boundary units per subject:", {p: int(members[p].sum()) for p in pats})

**Reference without a cut.** The dip and ramp tests are run, with the same groups, on traces aligned
to the NB virtual boundaries (the middle of clips with no cut), where the model predicts nothing. A
test that passes there as often as at cuts does not separate a dip from noise.

In [ ]:
for s in usable:
    t_, k_, _ = s.cuts("all")
    for b in BINS:
        unit_traces[(s.patient, "nb", b)] = C.unit_mean_rates(s, t_[k_ == 0], b)
rows = []
for selname in SELS:
    d = groups_for(selname)
    members = {p: x.bnd.values for p, x in d.groupby("patient", sort=False)}
    onset_m = {p: x.onset.values for p, x in d.groupby("patient", sort=False)}
    pats = [p for p in members if members[p].any() and onset_m[p].any()]
    for b in BINS:
        for sm in SMOOTH:
            for zref in ZREF:
                gm, gs, rm, rs, centers, _, _ = group_average(pats, "nb", b, sm, zref, members)
                m = C.dip_ramp(gm, rm, centers)
                rows.append(dict(selection=selname, bin_ms=int(1000 * b), smooth_ms=int(1000 * sm) if sm else 0, zref=zref,
                                 peak_ms=round(1000 * m["peak_t"]), lag_ms=round(m["lag_ms"]), depth=round(m["depth"], 3),
                                 dip=m["dip"], slope=round(m["slope"], 3), ramp=m["ramp"]))
fig3a_nb = pd.DataFrame(rows)
print(fig3a_nb.to_string())
print("NB readings with dip and ramp:", int((fig3a_nb.dip & fig3a_nb.ramp).sum()), "of", len(fig3a_nb),
      "; lag range", fig3a_nb.lag_ms.min(), "to", fig3a_nb.lag_ms.max(), "ms")

**Variant, not a reading of the paper.** To see whether the result depends on how few boundary
neurons the re-selection finds, each of the 14 subjects kept under reading (a) gets as many boundary
neurons as [41]'s Supplementary Table 2 counts for it (boundary plus event cells in the MTL), taken as
its units with the largest increase in spike count from the 1 s before to the 1 s after SB and HB cuts,
less the same increase at NB virtual boundaries; onset neurons likewise, ranked by the increase after
clip onset. The ranking is a choice made here; it is shown only to bound the outcome.

In [ ]:
def ranked_members(s, k_bnd, k_on):
    t, k, _ = s.cuts("all")
    inc_b, inc_o = [], []
    for sp in s.spikes:
        post = C.window_counts(sp, t, 0, 1); pre = C.window_counts(sp, t, -1, 0)
        inc_b.append((post - pre)[k > 0].mean() - (post - pre)[k == 0].mean())
        inc_o.append((C.window_counts(sp, s.clip_start, 0, 1) - C.window_counts(sp, s.clip_start, -1, 0)).mean())
    mb = np.zeros(len(s.spikes), bool); mb[np.argsort(inc_b)[::-1][:k_bnd]] = True
    mo = np.zeros(len(s.spikes), bool); mo[np.argsort(inc_o)[::-1][:k_on]] = True
    return mb, mo

by_pat = {s.patient: s for s in usable}
rank_b, rank_o = {}, {}
for p in included_a:
    rank_b[p], rank_o[p] = ranked_members(by_pat[p], int(t.loc[p, "cut_neurons"]), int(t.loc[p, "onset"]))
rows = []
for subset, pats_r in (("included", included_a), ("clock shown", [p for p in included_a if p in clock_ok])):
  for cs in CUTSETS + ("nb",):
    for b in BINS:
        for sm in SMOOTH:
            for zref in ZREF:
                gm2, gs2, rm2, rs2, c2, _, _ = group_average(pats_r, cs, b, sm, zref, rank_b)
                m = C.dip_ramp(gm2, rm2, c2)
                rows.append(dict(subset=subset, cuts=cs, bin_ms=int(1000 * b), smooth_ms=int(1000 * sm) if sm else 0, zref=zref,
                                 subjects=len(pats_r), peak_ms=round(1000 * m["peak_t"]), lag_ms=round(m["lag_ms"]),
                                 depth=round(m["depth"], 3), dip=m["dip"], slope=round(m["slope"], 3), ramp=m["ramp"]))
fig3a_rank = pd.DataFrame(rows)
print(fig3a_rank.to_string())
for (subset, cut_kind), part in fig3a_rank.assign(kind=np.where(fig3a_rank.cuts == "nb", "NB reference", "cuts")).groupby(["subset", "kind"]):
    print(f"{subset}, {cut_kind}: readings with dip and ramp {int((part.dip & part.ramp).sum())} of {len(part)}; "
          f"lag {part.lag_ms.min()} to {part.lag_ms.max()} ms; depth {part.depth.min()} to {part.depth.max()}")

**Result: blocked, methods_underspecified. The re-selection by the Methods of [41] finds 8 to 10 boundary and event cells in the 18 subjects, where Supplementary Table 2 of [41] counts 76, and its chance rate (0.02% for boundary and 0.25% for event cells) is about a tenth of the 2.11% and 2.26% of [41], so it is not the selection of [41] the paper used; the dataset names no boundary neuron and [41] gives only counts. Under the re-selection a dip and a ramp are present in all 64 readings over the 5 to 9 included subjects, with lags of 75 to 490 ms, within tolerance of 130 ms in 5 readings, and in none of 32 readings at NB virtual boundaries. The encoding clock is shown in only 7 of 18 sessions; over those, dip and ramp are present in 50 of 64 readings, with lags of 220 to 490 ms. The rank-matched variant shows dip and ramp in 16 of 16 readings (lags 190 to 430 ms) but also in 4 of 8 readings at NB virtual boundaries. None of this decides the claim.**

## fig3b-individual-subjects

> The dip-then-ramp of non-boundary neurons after a cut is seen in most individual subjects, though
> not all (P64CS has no ramp and TWH120 has no dip).

Per subject, its own z-scored non-boundary trace and its own boundary peak, with the dip and ramp
tests of fig3a, under the re-selection readings (with their few subjects) and the rank-matched variant
(the 14 subjects of reading (a)).

In [ ]:
def per_subject(patients, members, cs, b, sm, zref):
    out = []
    for p in patients:
        g, r, centers = subject_traces(p, cs, b, sm, zref, members[p])
        m = C.dip_ramp(g, r, centers)
        out.append(dict(patient=p, clock_shown=p in clock_ok, peak_ms=round(1000 * m["peak_t"]), dip_ms=round(1000 * m["dip_t"]),
                        dip=m["dip"], ramp=m["ramp"], both=m["dip"] and m["ramp"]))
    return pd.DataFrame(out)

rows = []
for label, pats_, mem in [("rank-matched variant", included_a, rank_b)] + [
        (sn, [p for p, x in groups_for(sn).groupby("patient", sort=False) if x.bnd.any() and x.onset.any()],
         {p: x.bnd.values for p, x in groups_for(sn).groupby("patient", sort=False)}) for sn in SELS]:
    for cs in CUTSETS:
        for b in BINS:
            for sm in SMOOTH:
                for zref in ZREF:
                    ps = per_subject(pats_, mem, cs, b, sm, zref)
                    get = lambda p, c: bool(ps.set_index("patient").loc[p, c]) if p in set(ps.patient) else None
                    rows.append(dict(selection=label, cuts=cs, bin_ms=int(1000 * b), smooth_ms=int(1000 * sm) if sm else 0,
                                     zref=zref, subjects=len(ps), both=int(ps.both.sum()),
                                     both_clock_shown=f"{int(ps[ps.clock_shown].both.sum())} of {int(ps.clock_shown.sum())}",
                                     P64CS_ramp=get("P64CS", "ramp"), TWH120_dip=get("TWH120", "dip")))
fig3b = pd.DataFrame(rows)
print(fig3b.to_string())
ps = per_subject(included_a, rank_b, "all", 0.010, 0.050, "whole").sort_values("dip_ms")
print()
print("rank-matched variant, all cuts, 10 ms, 50 ms smoothing, whole-window z, sorted by dip time:")
print(ps.to_string(index=False))

**Result: blocked with fig3a-dip-after-boundary, on which it depends (methods_underspecified). Under the re-selection at most 9 subjects are included and up to 9 show both a dip and a ramp, but P64CS is never included and TWH120 shows a dip in every reading. Under the rank-matched variant 9 to 13 of the 14 subjects show both, and P64CS has a ramp and TWH120 a dip in every reading, so neither named exception appears.**

## fig3c-onset-control

> Aligned to movie onset, onset-responsive neurons show a peak, but the remaining neurons (boundary
> neurons included) show no dip-then-ramp, so the pattern is specific to boundary neurons.

Onset neurons against all other neurons, aligned to clip onset, with the same pooling, z-scoring, bins
and window; the dip and ramp are taken from the onset peak.

In [ ]:
rows = []
for label, pats_, mem in [("rank-matched variant", included_a, rank_o)] + [
        (sn, [p for p, x in groups_for(sn).groupby("patient", sort=False) if x.bnd.any() and x.onset.any()],
         {p: x.onset.values for p, x in groups_for(sn).groupby("patient", sort=False)}) for sn in SELS]:
  for subset, pp_ in (("included", pats_), ("clock shown", [p for p in pats_ if p in clock_ok])):
    for b in BINS:
        for sm in SMOOTH:
            for zref in ZREF:
                gm3, gs3, rm3, rs3, c3, _, _ = group_average(pp_, "onset", b, sm, zref, mem)
                m = C.dip_ramp(gm3, rm3, c3)
                rows.append(dict(selection=label, subset=subset, bin_ms=int(1000 * b), smooth_ms=int(1000 * sm) if sm else 0, zref=zref,
                                 subjects=len(pp_), onset_units=int(sum(mem[p].sum() for p in pp_)),
                                 onset_peak_ms=round(1000 * m["peak_t"]), onset_peak_z=round(float(gm3[c3 > 0].max()), 2),
                                 depth=round(m["depth"], 3), dip=m["dip"], slope=round(m["slope"], 3), ramp=m["ramp"],
                                 rest_passes=m["dip"] and m["ramp"]))
fig3c = pd.DataFrame(rows)
print(fig3c.to_string())
for (lab, subset), part in fig3c.groupby(["selection", "subset"]):
    print(f"{lab}, {subset}: non-onset trace passes the dip-and-ramp test in {int(part.rest_passes.sum())} of {len(part)} readings; "
          f"onset peak {part.onset_peak_ms.min()} to {part.onset_peak_ms.max()} ms")

**Result: blocked with fig3a-dip-after-boundary, on which it depends (methods_underspecified). The onset re-selection finds 100 to 101 onset cells against the 70 of [41]'s table for these subjects, with a per-subject correlation of 0.15 to 0.17. Over the included subjects the non-onset trace passes the dip-and-ramp test in 6 of 8 readings for each re-selection and in 4 of 8 for the rank-matched variant; over the subjects whose encoding clock is shown, in 0 or 6 of 8 and in 0 of 8. The onset trace peaks 635 to 1995 ms after clip onset.**

## Claims not tested

The simulation claims (fig1d-basic-capacity-four, fig1d-chunking-retrieves-six,
fig2b-hierarchical-retrieval, figs2-delayed-hebbian-binding) and the derivation eq1-new-magic-number
use no recorded data, and the dataset holds nothing they need; the simulation code is not public
("will be made available in a Github repository at a later stage") and Table I did not survive in the
text read. The Figure 4 claims rest on recall tables printed in Miller and Selfridge 1950 and its
Danish and Hindi replications, which are in no data archive.

In [ ]:
print(f"notebook run time {time.time() - t_start:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| link-zheng-table-s2 | reproduced: all 19 subjects match Supplementary Table 2 of [41] on ID, age and sex |
| methods-subjects-included | partially reproduced: 14 of 18 kept under [41]'s counts, but the dataset has 19 subjects |
| fig3a-dip-after-boundary | blocked, methods_underspecified |
| fig3b-individual-subjects | blocked, methods_underspecified (depends on fig3a) |
| fig3c-onset-control | blocked, methods_underspecified (depends on fig3a) |
| five simulation and derivation claims | blocked, other |
| three Figure 4 claims | blocked, other |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The paper takes its boundary and onset neurons from
[41]; which units they are is in neither the dataset nor [41]. Re-selecting them by the Methods of [41]
finds far fewer boundary and event cells than [41] reports, so the Figure 3 claims are run under every
reading but not decided. The subject exclusion is tested on [41]'s per-subject counts.

**What was found about the dataset.** The units table has no classification of units, though [41]
classified them, so the paper's boundary and onset neurons cannot be identified. The spike times of
TWH138 (sub-19) are not on the clock of its trial tables, in the pinned and the latest versions. In
only 7 of the 18 other sessions does the population respond to clip onsets (and in 3 sessions it
responds to recognition images but not to clip onsets), so for 11 sessions the check does not show
that spikes and encoding table share a clock. TWH125, with no clip-onset cell in [41], has one of the
strongest onset responses here. The dataset has 565 units, all in the MTL, where [41] reports 580 MTL
neurons. The latest version (0.230530.1822) changes the spike times of at least one session (TWH120).